## Model Training and Prediction

This notebook trains specific model configurations exported from the backtest app.
It reads `prediction_models_{timestamp}.csv` to determine which brand/channel/model 
combinations to train, then generates predictions with optional bias correction.

### Workflow
1. Load model selection file from backtest export
2. Train each specified brand/channel/model combination
3. Save models to joblib and CSV
4. Generate predictions with bias correction
5. Save results

In [1]:
# notebook config
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

# packages
import pandas as pd
import numpy as np
from joblib import dump, load
from functools import partial

# functions
import functions.lr_models as lr
import functions.transform as tf

In [2]:
# config
config = {
    'run': '202609_3P',         # unique name of the training run
    'predict': '202609_3P',     # unique name of the prediction set
    'model_selection': 'model_selections_202609_3P_20260825_132820_backtest.csv',  # selected models from a backtest
}

# When True, the delivered forecast (mean/ci_lo/ci_hi/roas_*) is bias-corrected
# and the uncorrected values are kept as *_raw
APPLY_BIAS = True

# 202603_1P
# 202603_2P
# 202603_3P
# 202603_reflows
# 202604_dev
# 202604_mmm
# 202604_1P
# 202604_2P
# 202605_1P
# 202605_2P


In [3]:
# Pipeline map is now defined in lr_models module
from functions.lr_models import PIPELINE_MAP

# Load model selection file
df_selection = pd.read_csv(f"../data/train/{config['model_selection']}")

print(f"Loaded {len(df_selection)} model selections:")
df_selection

Loaded 18 model selections:


,brand,model,estimator,base_estimator,bias_pct,bias_weeks,params
0,brand us,Search | Conversion | Brand,PLS,PLS,8.667021,4,NaN
1,brand us,Search | Conversion | Non Brand,WLS,WLS,7.633641,4,NaN
2,brand us,Search | Conversion | PMAX,RLM_Tukey,RLM,23.753543,6,"{""norm"": ""tukey""}"
3,brand us,Social | Conversion | CVR ASC Volume,WLS,WLS,17.449120,6,NaN
4,brand us,Social | Conversion | CVR ASC Value,WLS,WLS,22.264890,6,NaN
5,brand us,Social | Conversion | CVR ASC Omni,WLS,WLS,18.909104,6,NaN
6,brand ca,Search | Conversion | Brand,WLS,WLS,-35.590210,4,NaN
7,brand ca,Search | Conversion | PMAX,PLS,PLS,NaN,3,NaN
8,brand ca,Search | Conversion | RSC,PLS,PLS,-8.664697,4,NaN
9,brand ca,Social | Conversion | CVR ASC Volume,PCA,PCA,12.386119,4,NaN


In [4]:
# Load training data
df_train = pd.read_csv(f"../data/train/train_{config['run']}.csv")

print(f"Training data shape: {df_train.shape}")
df_train.head()

Training data shape: (4736, 49)


,brand,model,weekstart,spend,nd,visits,promo_presidents_day,promo_friends_&_family,promo_easter_pre-peak,promo_easter_peak,...,promo_winter_sale_phase_3,promo_easter,promo_labor_day,promo_winter_sale_preview,discount_0_5,discount_0_4,discount_0_7,discount_0_6,discount_0_2,discount_0_3
0,brand us,Social | Awareness | AWE Social Meta,2025-02-02,12501.4601,118679.1845,13795.6329,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,brand us,Social | Awareness | AWE Social Meta,2025-02-09,13166.4100,125842.6389,14617.6805,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,brand us,Social | Awareness | AWE Social Meta,2025-02-16,17410.7301,173416.8917,20157.0183,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,brand us,Social | Awareness | AWE Social Meta,2025-02-23,14551.6800,139022.3327,16130.5997,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,brand us,Social | Awareness | AWE Social Meta,2025-03-02,22963.3000,161703.7629,19188.0692,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Train Selected Models

Train each brand/channel/model combination specified in the selection file.

In [5]:
# Train selected models using lr_models.train_selected_models()
df_models = lr.train_selected_models(df_train, df_selection)

print(f"\nTrained {len(df_models)} models")
df_models.drop(columns=['model_obj']).head(10)

Training PLS (PLS) for brand us / Search | Conversion | Brand...


/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encount

Training WLS (WLS) for brand us / Search | Conversion | Non Brand...


/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  r

Training RLM_Tukey (RLM) for brand us / Search | Conversion | PMAX with params {'norm': 'tukey'}...


...

Training OLS (OLS) for brand outlet / Social | Conversion | CVR ASC Value...


/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:847: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Training OLS (OLS) for brand outlet / Social | Conversion | CVR ASC Volume...

Trained 18 models


/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:847: RuntimeWarning: invalid value encountered in sqrt
  return self.resid / sigma / np.sqrt(1 - hii)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/Users/jeff.parks/Dev/revenue-forecasting/.venv/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


,brand,model,response,estimator,base_estimator,params,R2,R2_CV,n_obs,n_outliers,n_features,wls_k,SW_stat,SW_p,BP_stat,BP_p,DW,cooks_n,VIF
0,brand us,Search | Conversion | Brand,nd,PLS,PLS,NaN,0.9647,0.7856,108,3,44,NaN,0.982790,1.915803e-01,18.816064,0.000082,1.123240,6.0,inf
1,brand us,Search | Conversion | Non Brand,nd,WLS,WLS,NaN,0.6761,-0.0156,59,1,44,2.000,0.976697,3.262589e-01,19.782676,0.999391,1.247429,NaN,inf
2,brand us,Search | Conversion | PMAX,nd,RLM_Tukey,RLM,"{""norm"": ""tukey""}",0.9231,0.6489,85,1,44,NaN,0.975981,1.176844e-01,27.903146,0.972068,0.667169,NaN,inf
3,brand us,Social | Conversion | CVR ASC Volume,nd,WLS,WLS,NaN,0.8694,0.0620,59,0,44,2.000,0.632249,6.775009e-11,45.373659,0.414534,1.526748,NaN,inf
4,brand us,Social | Conversion | CVR ASC Value,nd,WLS,WLS,NaN,0.8602,-0.2169,54,0,44,2.000,0.668511,9.030365e-10,44.389006,0.455238,1.409884,NaN,inf
5,brand us,Social | Conversion | CVR ASC Omni,nd,WLS,WLS,NaN,0.8051,0.3444,54,0,44,2.000,0.685181,1.775949e-09,45.741555,0.399683,1.040628,NaN,inf
6,brand ca,Search | Conversion | Brand,nd,WLS,WLS,NaN,0.8161,0.5600,108,2,44,0.529,0.948752,4.522730e-04,13.027972,0.999999,1.539090,NaN,inf
7,brand ca,Search | Conversion | PMAX,nd,PLS,PLS,NaN,0.8775,0.5309,85,0,44,NaN,0.948173,1.867814e-03,3.119710,0.210167,1.278385,5.0,inf
8,brand ca,Search | Conversion | RSC,nd,PLS,PLS,NaN,0.9300,0.6749,85,1,44,NaN,0.962801,1.587011e-02,2.171711,0.337613,1.377756,3.0,inf
9,brand ca,Social | Conversion | CVR ASC Volume,nd,PCA,PCA,NaN,0.8722,0.4744,55,1,44,NaN,0.961755,8.258572e-02,8.224342,0.914474,1.905463,7.0,inf


In [6]:
# Save models
output_name = f"models_{config['run']}"

# Save with fitted model objects
dump(df_models, f"../data/output/{output_name}.joblib")

# Save stats only (no model objects)
df_models.drop(columns=['model_obj']).to_csv(f"../data/output/{output_name}.csv", index=False)

print(f"Saved to: data/output/{output_name}.joblib")
print(f"Saved to: data/output/{output_name}.csv")

Saved to: data/output/models_202609_3P.joblib
Saved to: data/output/models_202609_3P.csv


### Generate Predictions

Load prediction data and generate predictions with bias correction.

In [7]:
# Load prediction data
df_preds = pd.read_csv(f"../data/predict/predict_{config['predict']}.csv")
df_preds['weekstart'] = pd.to_datetime(df_preds['weekstart'])

# Aggregate by brand, channel, weekstart
df_preds = df_preds.groupby(['brand', 'model', 'weekstart'], as_index=False)[['spend']].sum()

# Merge promos
df_promos = pd.read_csv(f"../data/train/promo_dummies.csv")
df_promos['weekstart'] = pd.to_datetime(df_promos['weekstart'])
df_preds = df_preds.merge(df_promos, on=['weekstart', 'brand'], how='left')

print(f"Prediction data shape: {df_preds.shape}")
df_preds.head()

Prediction data shape: (175, 47)


/var/folders/sq/nh5_91qn4n7dggks6m77sgc40000gp/T/ipykernel_65193/725875647.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_preds['weekstart'] = pd.to_datetime(df_preds['weekstart'])


,brand,model,weekstart,spend,promo_presidents_day,promo_friends_&_family,promo_easter_pre-peak,promo_easter_peak,promo_spring_sale,promo_stylecash_redeem,...,promo_winter_sale_phase_3,promo_easter,promo_labor_day,promo_winter_sale_preview,discount_0_5,discount_0_4,discount_0_7,discount_0_6,discount_0_2,discount_0_3
0,brand us,Search | Conversion | Brand,2026-08-30,9372.662678,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,brand us,Search | Conversion | Brand,2026-09-06,9487.061001,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,brand us,Search | Conversion | Brand,2026-09-13,20469.300100,0,2,0,0,0,0,...,0,0,0,0,0,2,0,0,0,0
3,brand us,Search | Conversion | Brand,2026-09-20,25658.824060,0,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,brand us,Search | Conversion | Brand,2026-09-27,11011.822160,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Build bias dictionary from selection file
# Keys are (brand, channel, estimator) tuples, values are bias percentages
# Bias % comes from the backtest app's recent-weeks window (default: last 8 weeks)
BIAS_CLIP_PCT = 50  # guards the 1/(1 + bias/100) correction factor against extremes

bias_dict = {}
if APPLY_BIAS:
    for _, row in df_selection.iterrows():
        key = (row['brand'], row['model'], row['estimator'])
        bias_val = row.get('bias_pct', 0)
        if pd.notna(bias_val) and bias_val != 0:
            bias_dict[key] = float(np.clip(bias_val, -BIAS_CLIP_PCT, BIAS_CLIP_PCT))

    print(f"Bias corrections to apply (clipped to +/-{BIAS_CLIP_PCT}%):")
    for k, v in bias_dict.items():
        print(f"  {k[0]} / {k[1]} / {k[2]}: {v:.1f}%")
else:
    print("APPLY_BIAS is False - forecasts will be left uncorrected.")

if not bias_dict:
    print("  (none)")
    bias_dict = None


Bias corrections to apply (clipped to +/-50%):
  brand us / Search | Conversion | Brand / PLS: 8.7%
  brand us / Search | Conversion | Non Brand / WLS: 7.6%
  brand us / Search | Conversion | PMAX / RLM_Tukey: 23.8%
  brand us / Social | Conversion | CVR ASC Volume / WLS: 17.4%
  brand us / Social | Conversion | CVR ASC Value / WLS: 22.3%
  brand us / Social | Conversion | CVR ASC Omni / WLS: 18.9%
  brand ca / Search | Conversion | Brand / WLS: -35.6%
  brand ca / Search | Conversion | RSC / PLS: -8.7%
  brand ca / Social | Conversion | CVR ASC Volume / PCA: 12.4%
  brand ca / Social | Conversion | CVR ASC Value / PCA: 20.3%
  brand outlet / Search | Conversion | Brand / WLS: 3.2%
  brand outlet / Search | Conversion | Non Brand / WLS: 2.6%
  brand outlet / Search | Conversion | RSC / PLS: 7.4%
  brand outlet / Search | Conversion | PMAX / PLS: -2.8%
  brand outlet / Social | Conversion | CVR ASC Omni / OLS: 6.6%
  brand outlet / Social | Conversion | CVR ASC Value / OLS: 5.2%
  brand

In [9]:
# Generate predictions with bias correction
df_results = lr.pred_lr(df_models, df_preds, bias=bias_dict)

print(f"Generated {len(df_results)} predictions")
df_results.head()

Generated 90 predictions


,brand,model,weekstart,spend,response,estimator,base_estimator,params,bias,mean,mean_se,mean_ci_lower,mean_ci_upper,obs_ci_lower,obs_ci_upper,mean_adj,ci_lo_adj,ci_hi_adj
0,brand us,Search | Conversion | Brand,2026-08-30,9372.662678,nd,PLS,PLS,NaN,8.667021,7.254658e+05,12532.577332,7.046626e+05,7.462690e+05,5.501769e+05,9.007547e+05,6.676044e+05,6.484604e+05,6.867484e+05
1,brand us,Search | Conversion | Brand,2026-09-06,9487.061001,nd,PLS,PLS,NaN,8.667021,1.141415e+06,12927.154279,1.119956e+06,1.162873e+06,9.660466e+05,1.316782e+06,1.050378e+06,1.030631e+06,1.070125e+06
2,brand us,Search | Conversion | Brand,2026-09-13,20469.300100,nd,PLS,PLS,NaN,8.667021,1.822929e+06,20476.815179,1.788939e+06,1.856919e+06,1.645591e+06,2.000267e+06,1.677536e+06,1.646257e+06,1.708815e+06
3,brand us,Search | Conversion | Brand,2026-09-20,25658.824060,nd,PLS,PLS,NaN,8.667021,1.536586e+06,15537.600902,1.510795e+06,1.562377e+06,1.360635e+06,1.712537e+06,1.414032e+06,1.390297e+06,1.437766e+06
4,brand us,Search | Conversion | Brand,2026-09-27,11011.822160,nd,PLS,PLS,NaN,8.667021,7.655201e+05,12181.794370,7.452992e+05,7.857411e+05,5.902993e+05,9.407409e+05,7.044641e+05,6.858559e+05,7.230722e+05


In [10]:
# Prep results
cols_to_drop = [c for c in ['mean_se', 'obs_ci_lower', 'obs_ci_upper'] if c in df_results.columns]
if cols_to_drop:
    df_results = df_results.drop(cols_to_drop, axis=1)

# Rename CI columns for consistency
df_results = df_results.rename(columns={
    'mean_ci_lower': 'ci_lo',
    'mean_ci_upper': 'ci_hi'
})

# Promote bias-corrected values to the delivered forecast, keeping originals as *_raw
if APPLY_BIAS and 'mean_adj' in df_results.columns:
    for base, adj in [('mean', 'mean_adj'), ('ci_lo', 'ci_lo_adj'), ('ci_hi', 'ci_hi_adj')]:
        df_results[f'{base}_raw'] = df_results[base]
        df_results[base] = df_results[adj]
    df_results = df_results.drop(columns=['mean_adj', 'ci_lo_adj', 'ci_hi_adj'])

    n_corrected = int((df_results['bias'] != 0).sum())
    pct_change = (df_results['mean'] / df_results['mean_raw'] - 1) * 100
    print(f"Bias applied to {n_corrected} of {len(df_results)} rows")
    print(f"  mean change: {pct_change.mean():+.2f}%  "
          f"(min {pct_change.min():+.2f}%, max {pct_change.max():+.2f}%)")
else:
    print("No bias correction applied - forecast columns are uncorrected.")

# Calculate ROAS from the delivered forecast
df_results['roas_lo'] = df_results['ci_lo'] / df_results['spend']
df_results['roas_med'] = df_results['mean'] / df_results['spend']
df_results['roas_hi'] = df_results['ci_hi'] / df_results['spend']

# Round
df_results = df_results.round(4)

df_results.head()


Bias applied to 85 of 90 rows
  mean change: -3.54%  (min -19.19%, max +55.26%)


/var/folders/sq/nh5_91qn4n7dggks6m77sgc40000gp/T/ipykernel_65193/3498647350.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_results = df_results.round(4)


,brand,model,weekstart,spend,response,estimator,base_estimator,params,bias,mean,ci_lo,ci_hi,mean_raw,ci_lo_raw,ci_hi_raw,roas_lo,roas_med,roas_hi
0,brand us,Search | Conversion | Brand,2026-08-30,9372.6627,nd,PLS,PLS,NaN,8.667,6.676044e+05,6.484604e+05,6.867484e+05,7.254658e+05,7.046626e+05,7.462690e+05,69.1864,71.2289,73.2714
1,brand us,Search | Conversion | Brand,2026-09-06,9487.0610,nd,PLS,PLS,NaN,8.667,1.050378e+06,1.030631e+06,1.070125e+06,1.141415e+06,1.119956e+06,1.162873e+06,108.6355,110.7169,112.7983
2,brand us,Search | Conversion | Brand,2026-09-13,20469.3001,nd,PLS,PLS,NaN,8.667,1.677536e+06,1.646257e+06,1.708815e+06,1.822929e+06,1.788939e+06,1.856919e+06,80.4257,81.9538,83.4819
3,brand us,Search | Conversion | Brand,2026-09-20,25658.8241,nd,PLS,PLS,NaN,8.667,1.414032e+06,1.390297e+06,1.437766e+06,1.536586e+06,1.510795e+06,1.562377e+06,54.1840,55.1090,56.0340
4,brand us,Search | Conversion | Brand,2026-09-27,11011.8222,nd,PLS,PLS,NaN,8.667,7.044641e+05,6.858559e+05,7.230722e+05,7.655201e+05,7.452992e+05,7.857411e+05,62.2836,63.9734,65.6633


In [11]:
# Save results
results_name = f"results_{config['predict']}.csv"
df_results.to_csv(f"../data/output/{results_name}", index=False)

print(f"Saved to: data/output/{results_name}")

Saved to: data/output/results_202609_3P.csv


### Summary

In [12]:
# Summary
print("="*60)
print("TRAINING & PREDICTION COMPLETE")
print("="*60)
print(f"\nModels trained: {len(df_models)}")
print(f"Predictions generated: {len(df_results)}")
print(f"\nOutputs:")
print(f"  - data/output/{output_name}.joblib")
print(f"  - data/output/{output_name}.csv")
print(f"  - data/output/{results_name}")

TRAINING & PREDICTION COMPLETE

Models trained: 18
Predictions generated: 90

Outputs:
  - data/output/models_202609_3P.joblib
  - data/output/models_202609_3P.csv
  - data/output/results_202609_3P.csv
